<a href="https://colab.research.google.com/github/AmirMohammad73/semantic_folding/blob/main/Umap_fingerprint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scipy

In [2]:
!pip install umap-learn
!pip install Levenshtein

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 21.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
import umap
import os
import gensim.downloader as api

# Load corpus
with open("/content/drive/MyDrive/secondopenbook1.txt", "r") as file:
    corpus = file.readlines()

# Load GloVe word embeddings
glove_model = api.load("glove-wiki-gigaword-300")

# Create document vectors using GloVe embeddings
document_vectors = []
for doc in corpus:
    doc_vector = []
    for word in doc.split():
        if word in glove_model:
            doc_vector.append(glove_model[word])
    if doc_vector:
        document_vectors.append(np.mean(doc_vector, axis=0))
    else:
        document_vectors.append(np.zeros(300))

# Dimensionality reduction
n_components = 300
svd = TruncatedSVD(n_components=n_components)
document_vectors_reduced = svd.fit_transform(document_vectors)

# Clustering with UMAP
umap_model = umap.UMAP(n_components=2, random_state=123)  # Set a fixed random seed
document_vectors_umap = umap_model.fit_transform(document_vectors_reduced)

# Step 1: Get the range for the document_vectors_umap
x_min, x_max = document_vectors_umap[:, 0].min(), document_vectors_umap[:, 0].max()
y_min, y_max = document_vectors_umap[:, 1].min(), document_vectors_umap[:, 1].max()

# Step 2: Create a 16x16 grid to bin the data
num_bins = 16
x_edges = np.linspace(x_min, x_max, num_bins)
y_edges = np.linspace(y_min, y_max, num_bins)

# Step 3: Assign document ID to the respective bins (the result will be indices of the bins)
x_bin_indices = np.digitize(document_vectors_umap[:, 0], bins=x_edges) - 1 # -1 to convert to 0-indexed
y_bin_indices = np.digitize(document_vectors_umap[:, 1], bins=y_edges) - 1 # -1 to convert to 0-indexed

# Step 4: Make a list of 16x16 grid coordinates plus document IDs
mapped_data = list(zip(x_bin_indices, y_bin_indices, range(len(document_vectors_umap))))

# Convert to a numpy array if necessary
mapped_data_array = np.array(mapped_data)

# Initialize a fourth column in 'mapped_data_array' for the frequency counts, setting them initially to zero
mapped_data_array = np.hstack((mapped_data_array, np.zeros((mapped_data_array.shape[0], 1), dtype=int)))

In [4]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [5]:
from nltk.corpus import wordnet
import spacy
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import umap
import json

# Load spaCy's English model
nlp = spacy.load("en_core_web_sm")

# def standardize_synonyms(text):
#     words = text.split()
#     standardized_words = []
#     for word in words:
#         synsets = wordnet.synsets(word)
#         if synsets:
#             standardized_word = synsets[0].lemmas()[0].name()
#             standardized_words.append(standardized_word)
#         else:
#             standardized_words.append(word)
#     return " ".join(standardized_words)

# Load corpus
with open("/content/drive/MyDrive/cleaned_corpus_final3.txt", "r") as file:
    corpus = file.readlines()

# Tokenize the corpus
tokenized_corpus = [context.split() for context in corpus]

# Create the vocabulary set
vocabulary = set(word for context in tokenized_corpus for word in context)

# Initialize the document vectors
num_contexts = len(corpus)
num_vocab = len(vocabulary)
document_vectors = np.zeros((num_contexts, num_vocab), dtype=int)

# Fill in the document vectors
for i, context in enumerate(tokenized_corpus):
    for j, word in enumerate(vocabulary):
        if word in context:
            document_vectors[i, j] = 1

# Clustering with UMAP
umap_model = umap.UMAP(n_components=2)  # Set a fixed random seed
document_vectors_umap = umap_model.fit_transform(document_vectors)

# Step 1: Get the range for the document_vectors_umap
x_min, x_max = document_vectors_umap[:, 0].min(), document_vectors_umap[:, 0].max()
y_min, y_max = document_vectors_umap[:, 1].min(), document_vectors_umap[:, 1].max()

# Step 2: Create a 16x16 grid to bin the data
num_bins = 20
x_edges = np.linspace(x_min, x_max, num_bins)
y_edges = np.linspace(y_min, y_max, num_bins)

# Step 3: Assign document ID to the respective bins (the result will be indices of the bins)
x_bin_indices = np.digitize(document_vectors_umap[:, 0], bins=x_edges) - 1 # -1 to convert to 0-indexed
y_bin_indices = np.digitize(document_vectors_umap[:, 1], bins=y_edges) - 1 # -1 to convert to 0-indexed

# Step 4: Make a list of 16x16 grid coordinates plus document IDs
mapped_data = list(zip(x_bin_indices, y_bin_indices, range(len(document_vectors_umap))))

# Convert to a numpy array if necessary
mapped_data_array = np.array(mapped_data)

# Initialize a fourth column in 'mapped_data_array' for the frequency counts, setting them initially to zero
mapped_data_array = np.hstack((mapped_data_array, np.zeros((mapped_data_array.shape[0], 1), dtype=int)))


In [ ]:
# Load train.txt
with open("/content/drive/MyDrive/train_cleaned.txt", "r") as file:
    train_data_lines = file.readlines()

# Fine-tuning loop
num_epochs = 1
learning_rate = 0.1
n = 0
for epoch in range(num_epochs):
    for line in train_data_lines:
        n += 1
        if n%10 == 0:
            print(n)
        parts = line.strip().split(" // ")
        question = parts[0]
        choices = parts[1:-1]  # Exclude last part which is the correct answer

        correct_answer = parts[-1]

        # Create fingerprints for the question and answer choices
        question_vector = tfidf_vectorizer.transform([question])
        question_vector_reduced = svd.transform(question_vector)
        question_vector_umap = umap_model.transform(question_vector_reduced)
        answer_vectors = tfidf_vectorizer.transform(choices)
        answer_vectors_reduced = svd.transform(answer_vectors)
        answer_vectors_umap = umap_model.transform(answer_vectors_reduced)

        # Find the closest answer choice to the question
        distances = np.linalg.norm(question_vector_umap - answer_vectors_umap, axis=1)
        predicted_answer_index = np.argmin(distances)
        predicted_answer = choices[predicted_answer_index]

        # Update the semantic space if the prediction is incorrect
        if predicted_answer != correct_answer:
            # Move the correct answer closer to the question
            mapped_data_array = mapped_data_array.astype(float)  # Convert to float type
            mapped_data_array[predicted_answer_index, :2] += learning_rate * (question_vector_umap - answer_vectors_umap[predicted_answer_index]).reshape(-1)

            # Move the incorrect answer further away from the question
            correct_answer_indices = np.argwhere(np.array(choices) == correct_answer)
            if len(correct_answer_indices) > 0:  # Check if the array is not empty
                correct_answer_index = correct_answer_indices[0][0]
                mapped_data_array[correct_answer_index, :2] -= learning_rate * (question_vector_umap - answer_vectors_umap[correct_answer_index]).reshape(-1)

# Save the fine-tuned semantic space
np.save("/content/drive/MyDrive/fine_tuned_semantic_space.npy", mapped_data_array)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import umap

# Load corpus
with open("/content/drive/MyDrive/cleaned_corpus_final1.txt", "r") as file:
    corpus = file.readlines()

# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5988)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# Dimensionality reduction
n_components = 5988
svd = TruncatedSVD(n_components=n_components)
document_vectors = svd.fit_transform(tfidf_matrix)

# Clustering with UMAP
umap_model = umap.UMAP(n_components=2, random_state=123)  # Set a fixed random seed
document_vectors_umap = umap_model.fit_transform(document_vectors)
# Step 1: Get the range for the document_vectors_umap
x_min, x_max = document_vectors_umap[:, 0].min(), document_vectors_umap[:, 0].max()
y_min, y_max = document_vectors_umap[:, 1].min(), document_vectors_umap[:, 1].max()

# Step 2: Create a 16x16 grid to bin the data
num_bins = 16
x_edges = np.linspace(x_min, x_max, num_bins)
y_edges = np.linspace(y_min, y_max, num_bins)

# Step 3: Assign document ID to the respective bins (the result will be indices of the bins)
x_bin_indices = np.digitize(document_vectors_umap[:, 0], bins=x_edges) - 1 # -1 to convert to 0-indexed
y_bin_indices = np.digitize(document_vectors_umap[:, 1], bins=y_edges) - 1 # -1 to convert to 0-indexed

# Step 4: Make a list of 16x16 grid coordinates plus document IDs
mapped_data = list(zip(x_bin_indices, y_bin_indices, range(len(document_vectors_umap))))

# Convert to a numpy array if necessary
mapped_data_array = np.array(mapped_data)

# Initialize a fourth column in 'mapped_data_array' for the frequency counts, setting them initially to zero
mapped_data_array = np.hstack((mapped_data_array, np.zeros((mapped_data_array.shape[0], 1), dtype=int)))

In [6]:
import math
# Define a function to calculate the Euclidean distance between two matrices
def calculate_euclidean_distance(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Euclidean distance.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the Euclidean distance
    euclidean_distance = math.dist(matrix1_flat, matrix2_flat)
    return euclidean_distance

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_euclidean(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    euclidean_distances = []

    # Calculate the Euclidean distance of the question to each option
    for option_matrix in options_matrices:
        distance = calculate_euclidean_distance(question_matrix, option_matrix)
        euclidean_distances.append(distance)

    # Since we want similarity, not distance, we can invert the distances
    # The smaller the distance, the higher the similarity
    euclidean_similarities = [1 / distance if distance != 0 else float('inf') for distance in euclidean_distances]

    return euclidean_similarities

In [7]:
from scipy.spatial import distance
# Define a function to calculate the Hamming distance between two matrices
def calculate_hamming_distance(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Hamming distance.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the Hamming distance
    hamming_distance = distance.hamming(matrix1_flat, matrix2_flat)
    return hamming_distance

# Define a function to calculate the similarity based on Hamming distance
def calculate_hamming_similarity(matrix1, matrix2):
    hamming_distance = calculate_hamming_distance(matrix1, matrix2)
    hamming_similarity = 1 - (hamming_distance)
    return hamming_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_hamming(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    hamming_similarities = []

    # Calculate the Hamming similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_hamming_similarity(question_matrix, option_matrix)
        hamming_similarities.append(similarity)

    return hamming_similarities

In [8]:
# Define a function to calculate the Jaccard similarity between two matrices
def calculate_jaccard_similarity(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Jaccard similarity.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the intersection and union of the matrices
    intersection = np.sum(np.minimum(matrix1_flat, matrix2_flat))
    union = np.sum(np.maximum(matrix1_flat, matrix2_flat))

    # Calculate the Jaccard similarity
    jaccard_similarity = intersection / union if union != 0 else 0
    return jaccard_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_jaccard(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    jaccard_similarities = []

    # Calculate the Jaccard similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_jaccard_similarity(question_matrix, option_matrix)
        jaccard_similarities.append(similarity)

    return jaccard_similarities

In [9]:
from numpy.linalg import norm
# Define a function to calculate the Cosine similarity between two matrices
def calculate_cosine_similarity(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Cosine similarity.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the dot product and magnitudes of the matrices
    dot_product = np.dot(matrix1_flat, matrix2_flat)
    magnitude1 = norm(matrix1_flat)
    magnitude2 = norm(matrix2_flat)

    # Calculate the Cosine similarity
    cosine_similarity = dot_product / (magnitude1 * magnitude2) if magnitude1 != 0 and magnitude2 != 0 else 0
    return cosine_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_cosine(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    cosine_similarities = []

    # Calculate the Cosine similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_cosine_similarity(question_matrix, option_matrix)
        cosine_similarities.append(similarity)

    return cosine_similarities

In [10]:
from Levenshtein import distance as levenshtein_distance

# Define a function to calculate the Levenshtein distance between two matrices
def calculate_levenshtein_distance(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Levenshtein distance.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Convert the arrays to strings
    matrix1_str = ''.join(map(str, matrix1_flat))
    matrix2_str = ''.join(map(str, matrix2_flat))

    # Calculate the Levenshtein distance
    levenshtein_distance_value = levenshtein_distance(matrix1_str, matrix2_str)

    return levenshtein_distance_value

# Define a function to calculate the similarity based on Levenshtein distance
def calculate_levenshtein_similarity(matrix1, matrix2):
    levenshtein_distance_value = calculate_levenshtein_distance(matrix1, matrix2)
    max_distance = max(matrix1.size, matrix2.size)
    levenshtein_similarity = 1 - (levenshtein_distance_value / max_distance)
    return levenshtein_similarity

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_levenshtein(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    levenshtein_similarities = []

    # Calculate the Levenshtein similarity of the question to each option
    for option_matrix in options_matrices:
        similarity = calculate_levenshtein_similarity(question_matrix, option_matrix)
        levenshtein_similarities.append(similarity)

    return levenshtein_similarities

In [11]:
def calculate_sorensen_dice_index(matrix1, matrix2):
    # Ensure that both matrices have the same shape
    if matrix1.shape != matrix2.shape:
        raise ValueError("Matrices must have the same dimensions to calculate Sørensen-Dice index.")

    # Flatten the matrices to 1D arrays
    matrix1_flat = matrix1.flatten()
    matrix2_flat = matrix2.flatten()

    # Calculate the Sørensen-Dice index
    intersection = np.sum(np.minimum(matrix1_flat, matrix2_flat)) * 2
    total = np.sum(matrix1_flat) + np.sum(matrix2_flat)
    sorensen_dice_index = intersection / total if total != 0 else 0
    return sorensen_dice_index

# Assuming 'matrices' is a list of 5 matrices where the first matrix is the question
# and the next four are the options
def compare_question_to_options_sorensen_dice(matrices):
    question_matrix = matrices[0]
    options_matrices = matrices[1:]
    sorensen_dice_indices = []

    # Calculate the Sørensen-Dice index of the question to each option
    for option_matrix in options_matrices:
        index = calculate_sorensen_dice_index(question_matrix, option_matrix)
        sorensen_dice_indices.append(index)

    return sorensen_dice_indices

In [12]:
def matrix_to_sdr(matrix, num_largest):
    # Flatten the matrix into a 1D array
    flat_matrix = np.array(matrix).flatten()
    sorted_values = np.array(sorted(flat_matrix, reverse=True))
    temp_vector = flat_matrix
    result = np.zeros_like(flat_matrix)
    result_matrix = result.reshape(np.shape(matrix))  # Initialize result_matrix
    for i in range(num_largest):
        if sorted_values[i] == 0:
            break
        result[np.where(temp_vector == sorted_values[i])[0][0]] = 1
        temp_vector[np.where(temp_vector == sorted_values[i])[0][0]] = -1
        result_matrix = result.reshape(np.shape(matrix))
    return result_matrix


In [13]:
# prompt: Write a function that returns the index of the largest value of an array as letters. That is, if the first element has the largest value, it returns the letter A. If the second element has the highest value, B and so on.

def largest_value_index_to_letter(array):
    # Find the index of the maximum value
    max_index = np.argmax(array)

    # Convert the index to a letter
    # A = 0, B = 1, C = 2, ...
    letter = chr(ord('A') + max_index)

    return letter

In [20]:
n = 1
total_questions = 500
euclidean_correct = 0
hamming_correct = 0
jaccard_correct = 0
cosine_correct = 0
levenshtein_correct = 0
sorensen_correct = 0
# Read input from the file
with open("/content/drive/MyDrive/test_cleaned3.txt", "r") as file:
    for line in file:
        parts = line.strip().split(" // ")
        question, option1, option2, option3, option4, answer = parts
        # Initialize a list to store the matrices for each input
        matrices = []

        # Initialize a frequency counter for each document
        word_frequencies_list = []
        for user_input in [question, option1, option2, option3, option4]:
            word_frequencies = {doc_id: 0 for doc_id in range(len(corpus))}
            # Scan through the corpus to find the exact frequency of each word in each document
            for doc_id, document in enumerate(corpus):
                # Convert the document to lowercase and split into words
                words = document.lower().split()
                # Count the occurrences of each word in the words list
                for word in user_input.split():
                    word_frequencies[doc_id] += words.count(word.lower())
            word_frequencies_list.append(word_frequencies)

        # Update 'mapped_data_array' to include the total word count for each input
        for word_frequencies in word_frequencies_list:
            for i, (x_index, y_index, doc_id) in enumerate(mapped_data):
                total_word_count = word_frequencies[doc_id]
                mapped_data_array[i, 3] = total_word_count

            # Optionally, filter out the entries with a total word count of zero
            filtered_mapped_data = mapped_data_array[mapped_data_array[:, 3] > 0]
            # Initialize the 24x24 matrix with zeros
            matrix = np.zeros((num_bins, num_bins), dtype=int)

            # Iterate over the filtered_mapped_data to populate the matrix
            for data in filtered_mapped_data:
                x_coord, y_coord, doc_id, total_word_count = data
                matrix[int(y_coord), int(x_coord)] += int(total_word_count)  # Note that y_coord is used for rows, x_coord for columns
            matrix = matrix_to_sdr(matrix, math.floor(0.06*num_bins**2))
            matrices.append(matrix)

        # Calculate similarities and print the result
        euclidean_similarities = largest_value_index_to_letter(compare_question_to_options_euclidean(matrices))
        hamming_similarities = largest_value_index_to_letter(compare_question_to_options_hamming(matrices))
        jaccard_similarities = largest_value_index_to_letter(compare_question_to_options_jaccard(matrices))
        cosine_similarities = largest_value_index_to_letter(compare_question_to_options_cosine(matrices))
        levenshtein_distances = largest_value_index_to_letter(compare_question_to_options_levenshtein(matrices))
        sorensen_dice_indices = largest_value_index_to_letter(compare_question_to_options_sorensen_dice(matrices))

        # Calculate the correct percentage for each similarity method
        euclidean_correct += 1 if euclidean_similarities == answer else 0
        hamming_correct += 1 if hamming_similarities == answer else 0
        jaccard_correct += 1 if jaccard_similarities == answer else 0
        cosine_correct += 1 if cosine_similarities == answer else 0
        levenshtein_correct += 1 if levenshtein_distances == answer else 0
        sorensen_correct += 1 if sorensen_dice_indices == answer else 0
        # print similarities
        print(f"{n} {answer} {euclidean_similarities} {hamming_similarities} {jaccard_similarities} {cosine_similarities} {levenshtein_distances} {sorensen_dice_indices}")
        n += 1
print(f"euclidean similarity: {euclidean_correct/total_questions*100}%")
print(f"hamming similarity: {hamming_correct/total_questions*100}%")
print(f"jaccard similarity: {jaccard_correct/total_questions*100}%")
print(f"cosine similarity: {cosine_correct/total_questions*100}%")
print(f"leveneshtein similarity: {levenshtein_correct/total_questions*100}%")
print(f"sorensen similarity: {sorensen_correct/total_questions*100}%")

1 B C C C C C C
2 A A A D D B D
3 C C C B B C B
4 C A A A A A A
5 C C C C C C C
6 C B B D D B D
7 C D D A A C A
8 B A A A A A A
9 D B B B B B B
10 B A A A A A A
11 C D D B B D B
12 B D D C C D C
13 C C C A A B A
14 A A A B A A B
15 C B B B B B B
16 D C C C C B C
17 C C C C C C C
18 C C C B B C B
19 A A A D D A D
20 B C C A A D A
21 A A A A A A A
22 A A A A A A A
23 A A A A A A A
24 B C C C C A C
25 B A A A A A A
26 B A A B B A B
27 D B B A A B A
28 A A A A A A A
29 D C C C C C C
30 B A A A A A A
31 C B B C C B C
32 D C C B B D B
33 B D D A A C A
34 A D D D D B D
35 B C C C C C C
36 A B B B B C B
37 A D D A A D A
38 D A A A A A A
39 D A A A A D A
40 A C C D D C D
41 D B B C C B C
42 D B B D D B D
43 A A A A A C A
44 D D D A D D A
45 A C C C C C C
46 A A A A A B A
47 A C C C C C C
48 A C C A A A A
49 B C C A A C A
50 C A A A A C A
51 A B B B B B B
52 D A A A A A A
53 A C C C C C C
54 D D D B B B B
55 C B B A A B A
56 A B B B B B B
57 C B B A A A A
58 D B B B B B B
59 B C C C C C C
60 A C

In [23]:

# Prompt the user for a question and its four options
inputs = ['water', 'liquid', 'food', 'fruit', 'hot']
# for i in range(5):
#     user_input = input("Enter question or option {}: ".format(i + 1))
#     inputs.append(user_input)

# Initialize a list to store the matrices for each input
matrices = []

# Initialize a frequency counter for each document with a dict comprehension
word_frequencies_list = []
for user_input in inputs:
    word_frequencies = {doc_id: 0 for doc_id in range(len(corpus))}
    # Scan through the corpus to find the exact frequency of each word in each document
    for doc_id, document in enumerate(corpus):
        # Convert the document to lowercase and split into words
        words = document.lower().split()
        # Count the occurrences of each word in the words list
        for word in user_input.split():
            word_frequencies[doc_id] += words.count(word.lower())
    word_frequencies_list.append(word_frequencies)

# Update 'mapped_data_array' to include the total word count for each input
for word_frequencies in word_frequencies_list:
    for i, (x_index, y_index, doc_id) in enumerate(mapped_data):
        total_word_count = word_frequencies[doc_id]
        mapped_data_array[i, 3] = total_word_count

    # Optionally, filter out the entries with a total word count of zero
    filtered_mapped_data = mapped_data_array[mapped_data_array[:, 3] > 0]

    # Initialize the 24x24 matrix with zeros
    matrix = np.zeros((num_bins, num_bins), dtype=int)

    # Iterate over the filtered_mapped_data to populate the matrix
    for data in filtered_mapped_data:
        x_coord, y_coord, doc_id, total_word_count = data
        matrix[y_coord, x_coord] += total_word_count  # Note that y_coord is used for rows, x_coord for columns
    # matrix = matrix_to_sdr(matrix, math.floor(0.02*num_bins**2))
    # print(matrix)
    # Append the matrix to the list of matrices
    matrices.append(matrix)

# Calculate similarities and print the result
euclidean_similarities = compare_question_to_options_euclidean(matrices)
hamming_similarities = compare_question_to_options_hamming(matrices)
jaccard_similarities = compare_question_to_options_jaccard(matrices)
cosine_similarities = compare_question_to_options_cosine(matrices)
levenshtein_distances = compare_question_to_options_levenshtein(matrices)
sorensen_dice_indices = compare_question_to_options_sorensen_dice(matrices)

print(f"euclidean similarity: {euclidean_similarities}")
print(f"hamming similarity: {hamming_similarities}")
print(f"jaccard similarity: {jaccard_similarities}")
print(f"cosine similarity: {cosine_similarities}")
print(f"leveneshtein similarity: {levenshtein_distances}")
print(f"sorensen similarity: {sorensen_dice_indices}")
# Write the matrices to a file
with open("/content/drive/MyDrive/UMAP_output.txt", "w") as f:
    for i, matrix in enumerate(matrices):
        # Write the input number
        f.write("Matrix for input {}:\n".format(i + 1))
        # Write each element of the matrix individually
        for row in matrix:
            for element in row:
                f.write("{:4d}".format(element))  # Adjust the format to your preference
            f.write("\n")
        f.write("\n")

euclidean similarity: [0.0027207887861193025, 0.0023097891539375862, 0.002678790327828163, 0.0027295418972539663]
hamming similarity: [0.8125, 0.81, 0.825, 0.8125]
jaccard similarity: [0.05139186295503212, 0.036490683229813664, 0.009400705052878966, 0.048167539267015703]
cosine similarity: [0.18128283831812017, 0.018698288835198345, 0.055908812924267345, 0.21111164997622645]
leveneshtein similarity: [0.8225, 0.795, 0.8325, 0.815]
sorensen similarity: [0.09775967413441955, 0.07041198501872659, 0.018626309662398137, 0.0919080919080919]
